# Week 3 recap — Sheet 01 SOLUTIONS: grain, and the joins that multiply

Executed in the lab image. Every quoted number is what it actually printed.

If you take one thing from the week, take question 6: the same join, over the
same rows, gives a correct total and a wrong count.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Week 3 recap, sheet 01 — Grain and fan-out. Run this once.
import glob
import pandas as pd

BRONZE = "data/bronze/"
orders_raw = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(BRONZE + "orders_*.csv"))],
                       ignore_index=True)
customers = pd.read_csv(BRONZE + "customers.csv")
products = pd.read_csv(BRONZE + "products.csv")
returns = pd.read_csv(BRONZE + "returns.csv")

print("orders_raw", orders_raw.shape)
print("customers ", customers.shape)
print("products  ", products.shape)
print("returns   ", returns.shape)

PART A — state the grain before you touch anything

### Question 1

For each of the four tables, print its row count, its candidate key, the distinct count of that key, and whether it is unique. Then write, in one sentence each, what one row means.
> **NOTE:** day 4 worksheet 02 established that a grain is a *uniqueness claim*. A claim is checkable.

In [ ]:
checks = [("orders_raw", orders_raw, "LineID"),
          ("orders_raw", orders_raw, "OrderID"),
          ("customers", customers, "CustomerID"),
          ("products", products, "ProductID"),
          ("returns", returns, "OrderID")]
for name, df, key in checks:
    print("  %-11s %-12s rows %5d  distinct %5d  unique %s"
          % (name, key, len(df), df[key].nunique(), df[key].is_unique))
print()
print("  orders   : one row per order LINE   (but see below)")
print("  customers: one row per customer")
print("  products : one row per product")
print("  returns  : one row per returned ORDER")

```
  orders_raw  LineID       rows  8100  distinct  8060  unique False
  orders_raw  OrderID      rows  8100  distinct  5361  unique False
  customers   CustomerID   rows  1832  distinct  1832  unique True
  products    ProductID    rows  1234  distinct  1234  unique True
  returns     OrderID      rows   558  distinct   558  unique True
```

Five lines, and they decide every join on this sheet.

**The two dimensions are clean.** `CustomerID` and `ProductID` are unique, which
is exactly the property that makes joining to them safe — one match per fact row,
no multiplication.

**`returns.OrderID` is unique too**, one row per returned order. Hold that: it is
why question 3's `validate="many_to_one"` passes and question 4 still goes wrong.

**Both `orders` keys fail**, for different reasons. `OrderID` repeating is the
grain — an order has several lines, which is correct. `LineID` repeating is a
defect, and question 2 deals with it.

The habit this recaps: **run these five lines before writing a single `merge`.**
Every join failure in week 3 — day 1 worksheet 16, day 4 worksheets 02, 04 and 07
— was visible in a table like this before any code ran.

### Question 2

`LineID` is supposed to identify a line and does not. Count the offenders, show two of them side by side, and say whether they are corruption or something else.
> **NOTE:** day 1 worksheet 17 answered this. Confirm it rather than recalling it.

In [ ]:
dupes = orders_raw[orders_raw.LineID.duplicated(keep=False)]
print("rows with a repeated LineID:", len(dupes))
print("distinct LineIDs involved: ", dupes.LineID.nunique())
print("identical in every column: ", int(dupes.duplicated().sum()), "of", len(dupes))
print()
one = dupes[dupes.LineID == dupes.LineID.iloc[0]]
print(one[["LineID", "OrderID", "OrderDate", "Sales"]].to_string(index=False))
print()
orders = orders_raw.drop_duplicates(subset="LineID").copy()
print("after dedup:", len(orders), "rows -- use THIS below")
print("SUM(Sales) before dedup: %13.2f" % orders_raw.Sales.sum())
print("SUM(Sales) after dedup:  %13.2f" % orders.Sales.sum())

```
rows with a repeated LineID: 80
distinct LineIDs involved:  40
identical in every column:  40 of 80

 LineID  OrderID  OrderDate  Sales
   2381    52833 2011-12-24 317.82
   2381    52833 2011-12-24 317.82

after dedup: 8060 rows -- use THIS below
SUM(Sales) before dedup:   13633504.89
SUM(Sales) after dedup:    13570810.63
```

Eighty rows, forty `LineID`s, **identical in every column** — and dated
2011-12-24, in the last week of 2011.

Not corruption. A **re-delivery**: the 2012 extract re-sent the tail of 2011.
Day 1 worksheet 17 found this, and the reason it matters is the two totals at
the bottom. **13,633,504.89 against 13,570,810.63** — the duplicates carry
62,694.27 of revenue that does not exist, **0.462%**.

That percentage is the point. Too small to notice, too small to fail any
tolerance, wrong forever, and it grows every time a delivery overlaps.

The distinction to keep: identical copies are a **dedup rule**. Copies that
differ on the same key are an **escalation**, because someone has to decide which
is true. One line of code tells you which you have:

```python
dupes.duplicated().sum()   # 40 of 80 -> identical -> safe to drop
```

PART B — which joins are safe

### Question 3

Join `orders` to `customers` and to `products`, both with `validate="many_to_one"`. Print the row count after each. Then attempt the same `validate` on a join to `returns` and say why it behaves differently.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")
for name, dim, key in [("customers", customers, "CustomerID"),
                       ("products", products, "ProductID"),
                       ("returns", returns, "OrderID")]:
    before = len(orders)
    j = orders.merge(dim, on=key, how="left", validate="many_to_one")
    print("  + %-10s %5d -> %5d rows   (validate=many_to_one passed)"
          % (name, before, len(j)))
print()
print("all three pass: each key is unique on the RIGHT side.")
print("returns.OrderID unique:", returns.OrderID.is_unique)

```
  + customers   8060 ->  8060 rows   (validate=many_to_one passed)
  + products    8060 ->  8060 rows   (validate=many_to_one passed)
  + returns     8060 ->  8060 rows   (validate=many_to_one passed)

all three pass: each key is unique on the RIGHT side.
returns.OrderID unique: True
```

All three pass, all three preserve the row count. `validate="many_to_one"`
asserts the key is unique on the **right** side, and it is: one customer per
`CustomerID`, one product per `ProductID`, one return per `OrderID`.

This is the assertion worth putting on every fact-to-dimension join you write. It
converts the most common silent pipeline bug — a duplicated dimension row
inflating totals by a few plausible percent — into an immediate, named failure.

But read the last line carefully, because question 4 turns on it. **`returns`
passed.** Its key really is unique, and a `left` join to it really does preserve
the 8,060 rows.

So `validate=` has told you the truth and told you nothing about whether the join
answers your question. It checks the **shape**. Question 4 is about the
**meaning**.

### Question 4

So `many_to_one` passes for `returns` too. Show that this does **not** mean the join is harmless: count the rows an inner join to `returns` produces, and compare with `len(returns)`.
> **NOTE:** `validate=` checks the *shape* of the join. It cannot check the *question* you are asking.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")
r = orders.merge(returns, on="OrderID", how="inner")
print("orders rows: ", len(orders))
print("returns rows:", len(returns))
print("joined rows: ", len(r))
print()
print("distinct OrderID in the result:", r.OrderID.nunique())
print("rows per returned order: %.2f" % (len(r) / len(returns)))
print()
print("the RETURNS side fanned out, not orders:")
print("  every order line appears at most once:", r.LineID.is_unique)

```
orders rows:  8060
returns rows: 558
joined rows:  837

distinct OrderID in the result: 558
rows per returned order: 1.50

the RETURNS side fanned out, not orders:
  every order line appears at most once: True
```

**558 returns went in and 837 rows came out**, and `validate="many_to_one"`
passed on the same join in question 3.

Both facts are true because the fan-out is on the side nobody was watching. A
`left` join keeps 8,060 rows (question 3). An `inner` join keeps only the matched
ones — 837 — and each of the 558 returns is repeated once per line of its order,
1.50 on average.

**`orders` did not multiply.** `every order line appears at most once: True` —
each line matched at most one return, because `returns.OrderID` is unique. It is
the `returns` rows that were duplicated.

That direction is easy to get backwards and it decides everything downstream:
which measures survive (question 6) and which do not (question 5).

The check that reveals it in one line: **compare `nunique()` on the key you care
about with what it was before.** 558 in, 558 distinct out, 837 rows — so the
rows multiplied while the orders did not.

### Question 5

Answer *"how many orders were returned?"* three ways from that join — `len()`, `nunique()` on `OrderID`, and the truth from `returns` — and print all three.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")
r = orders.merge(returns, on="OrderID", how="inner")
print("len(joined)                  %5d   <- wrong" % len(r))
print("joined.OrderID.nunique()     %5d" % r.OrderID.nunique())
print("len(returns)                 %5d   <- the truth" % len(returns))
print()
print("overstated by %.0f%%" % (100 * (len(r) / len(returns) - 1)))

```
len(joined)                    837   <- wrong
joined.OrderID.nunique()       558
len(returns)                   558   <- the truth

overstated by 50%
```

Three answers to one question, and the obvious one is wrong by **50%**.

`len()` counts rows, and after the join a row is not a return — it is a
*return × line* pair. `nunique()` on `OrderID` recovers the right number here,
and `len(returns)` was right all along without any join at all.

Note that `nunique()` only works because you knew to ask for it. That is the
weakness of fixing this at query time: it depends on every analyst remembering
what the row means, forever. Day 4's answer was to make the fact table's grain a
promise so `COUNT(*)` is always safe; this sheet's answer is question 6's — do
not create the fan-out in the first place.

**The tell to recognise:** a count that lands between your two input row counts
and looks entirely plausible. 837 is bigger than 558 and smaller than 8,060. It
will grow month over month. Nothing about it looks wrong.

PART C — the twist

### Question 6

Now sum `Sales` over that same join, and compare with flagging returns using `isin` instead. Print both, and explain in one sentence why the count was wrong but the sum is not.
> **NOTE:** this is the sharpest idea of the week. Ask which *table* each number comes from.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID").copy()
r = orders.merge(returns, on="OrderID", how="inner")
orders["IsReturned"] = orders.OrderID.isin(set(returns.OrderID))

print("SUM(Sales) via the merge: %13.2f" % r.Sales.sum())
print("SUM(Sales) via isin:      %13.2f"
      % orders.loc[orders.IsReturned, "Sales"].sum())
print("identical:", round(r.Sales.sum(), 2)
      == round(orders.loc[orders.IsReturned, "Sales"].sum(), 2))
print()
print("rows: merge %d, isin %d (unchanged)" % (len(r), len(orders)))
print()
print("Sales lives on ORDERS, and each order line appears once in the join,")
print("so it is neither duplicated nor dropped. The COUNT came from RETURNS,")
print("which WAS duplicated.")

```
SUM(Sales) via the merge:    1485707.73
SUM(Sales) via isin:         1485707.73
identical: True

rows: merge 837, isin 8060 (unchanged)
```

**Identical to the cent** — and the count from the same join was 50% wrong.

This is the sharpest idea of week 3, and it is one sentence:

> A fan-out damages a measure only if the measure comes from the side that
> fanned out.

`Sales` lives on `orders`, at line grain. Each line appears **once** in the join,
because `returns.OrderID` is unique, so `Sales` is neither duplicated nor
dropped. Summing it is safe. The row count came from `returns`, which *was*
duplicated 558 → 837.

Same join. Same 837 rows. One correct number, one wrong one.

Which means "is this join safe?" is not a question with a yes/no answer. The
honest question is **"safe for which column?"** — and answering it takes ten
seconds once you know to ask.

The `isin` version is still the better code, not because the merge is wrong but
because it **cannot** be wrong:

```python
orders["IsReturned"] = orders.OrderID.isin(set(returns.OrderID))
```

8,060 rows in, 8,060 out, by construction. No later reader has to reason about
fan-out to trust a total. **When you need a flag, flag. When you need columns,
join.**

### Question 7

Generalise it. Take two measures — `Sales` (from orders) and a constant `1` standing for "a return" (from returns) — and sum both over the join and over the correct source. Print a small table showing which survives.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")
r = orders.merge(returns.assign(ReturnCount=1), on="OrderID", how="inner")

rows = [
    ("Sales", "orders", r.Sales.sum(),
     orders.loc[orders.OrderID.isin(set(returns.OrderID)), "Sales"].sum()),
    ("ReturnCount", "returns", float(r.ReturnCount.sum()), float(len(returns))),
]
print("%-13s %-9s %15s %15s %s" % ("MEASURE", "FROM", "OVER JOIN", "TRUE", ""))
for name, src, over, true in rows:
    ok = "ok" if round(over, 2) == round(true, 2) else "WRONG"
    print("%-13s %-9s %15.2f %15.2f %s" % (name, src, over, true, ok))
print()
print("the measure from the fanned-out side is the one that breaks.")

```
MEASURE       FROM            OVER JOIN            TRUE
Sales         orders         1485707.73      1485707.73 ok
ReturnCount   returns            837.00          558.00 WRONG

the measure from the fanned-out side is the one that breaks.
```

The whole sheet in three lines.

Two measures, one join, one table. `Sales` comes from `orders` — the side that
kept its grain — and survives exactly. `ReturnCount` comes from `returns` — the
side that was duplicated — and is inflated by the same 1.50 factor as the rows.

This is the diagnostic to carry out of week 3. Given any number computed after a
join, ask **two** questions:

1. Which table did this column come from?
2. Did that table's rows get duplicated by the join?

If the answer to (2) is yes, the number is wrong by the fan-out factor. If no, it
is fine — regardless of how alarming the row count looks.

It generalises past `SUM`. `COUNT`, `AVG` and `COUNT(DISTINCT)` all inherit the
same rule, and `AVG` is the nastiest of them: an average over fanned-out rows is
weighted by how many lines each order happened to have, which is a weighting
nobody chose and nobody can see.

PART D — the filter you did not write

### Question 8

Build a customer-enriched table two ways — `how="inner"` and `how="left"` — and print the row count of each. Then do the reverse direction with `indicator=True` and count customers who never ordered.
> **NOTE:** day 4 worksheet 07's `AssertionError: fact table has 2219 rows` was exactly this mistake.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")
for how in ("inner", "left"):
    print("  how=%-6s %5d rows" % (how, len(orders.merge(customers, on="CustomerID", how=how))))
print()
back = customers.merge(orders[["CustomerID"]].drop_duplicates(),
                       on="CustomerID", how="left", indicator=True)
print(back._merge.value_counts().to_string())
print()
print("customers who never ordered:", int((back._merge == "left_only").sum()))
print("(inner and left agree here only because every order HAS a customer)")

```
  how=inner   8060 rows
  how=left    8060 rows

_merge
both          1812
left_only       20
right_only       0

customers who never ordered: 20
```

`inner` and `left` agree at 8,060 — and that agreement is **evidence, not
safety**. It proves every order line has a customer. It would stop being true the
moment one did not, and the row count would silently go *down*, which reads as
"less data" rather than "a bug".

Day 4 worksheet 07's `AssertionError: fact table has 2219 rows` was exactly that
failure: an inner join, a table that passed every internal check — unique grain,
no nulls — and 156 rows missing that no check inside the table could find.

**An inner join is a filter you did not write down.** Use `left` when the left
table defines the row count, which in a pipeline it almost always does.

From the other direction, **20 customers have never ordered**. That is not a bug
— it is a lead list. The asymmetry is the normal shape of a fact-to-dimension
relationship:

- `left_only` from the **fact** side → broken pipeline, should be zero
- `left_only` from the **dimension** side → a business fact, count it

`indicator=True` answers both and costs nothing.

### Question 9

Write the three checks you would run after any join, as a small function, and run it on the customer join. Print a PASS/FAIL line for each.
> **NOTE:** row count unchanged, grain still unique, no unmatched rows on the fact side. Three lines, every join.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")


def check_join(before, after, grain, dim_col):
    return [
        ("row count unchanged", len(before), len(after)),
        ("grain still unique", True, bool(after[grain].is_unique)),
        ("no unmatched fact rows", 0, int(after[dim_col].isna().sum())),
    ]


joined = orders.merge(customers, on="CustomerID", how="left",
                      validate="many_to_one")
failed = 0
for name, want, got in check_join(orders, joined, "LineID", "CustomerSegment"):
    ok = want == got
    failed += not ok
    print("  %-24s expected %-6s got %-6s %s"
          % (name, want, got, "PASS" if ok else "FAIL"))
print()
print("%d of 3 passed" % (3 - failed))

```
  row count unchanged      expected 8060   got 8060   PASS
  grain still unique       expected True   got True   PASS
  no unmatched fact rows   expected 0      got 0      PASS

3 of 3 passed
```

Three checks, and between them they catch every join failure on this sheet:

**Row count unchanged** catches a fan-out — the dimension key was not unique, or
you joined to something at a coarser grain.

**Grain still unique** catches the same thing from the other direction, and is
the one that survives a refactor: if `LineID` stops being unique, the table has
stopped meaning what its documentation says.

**No unmatched fact rows** catches a broken foreign key. A null in a column that
came from the *right* side of a left join means the join found no match — and it
is invisible in the row count, because a left join keeps the row either way.

Wrapping them in a function is the point. Three assertions written once and
called after every join is a different discipline from remembering to eyeball a
row count, and it is the difference between a pipeline that fails loudly and one
that publishes a number that is 0.462% wrong.

Two worth adding in production: compare against the **previous run** (absolutes
pass happily on a source that halved), and reconcile a **control total** against
something computed outside the table — day 1 worksheet 17's bronze-vs-silver
check.

### Question 10

Finally, assert the claim that would have prevented all of this: merge `orders` and `returns` on `OrderID` with `validate="one_to_one"`. **This is supposed to fail.** Say which side breaks it and why.

In [ ]:
orders = orders_raw.drop_duplicates(subset="LineID")
print("orders  : %5d rows, %5d distinct OrderID -> unique %s"
      % (len(orders), orders.OrderID.nunique(), orders.OrderID.is_unique))
print("returns : %5d rows, %5d distinct OrderID -> unique %s"
      % (len(returns), returns.OrderID.nunique(), returns.OrderID.is_unique))
print()
print(orders.merge(returns, on="OrderID", validate="one_to_one").shape)

```
orders  :  8060 rows,  5361 distinct OrderID -> unique False
returns :   558 rows,   558 distinct OrderID -> unique True

MergeError: Merge keys are not unique in left dataset; not a one-to-one merge
Duplicates in left:
  OrderID
   37537
   37537
...
```

**The left side breaks it.** `returns` is unique on `OrderID`; `orders` is not —
5,361 distinct across 8,060 rows, because an order has several lines.

pandas refuses before producing a single row and names the offending keys. That
is question 4's fan-out, caught in advance by one keyword argument, and it is the
same error day 1 worksheet 16 and day 4 worksheet 04 both end on.

**What `validate=` is worth, precisely.** It would not have saved question 5:
`many_to_one` passes on this join (question 3), because the *shape* is legal. The
error there was asking a fanned-out result to count something. So:

- `validate=` protects you from a join you did not mean to write
- **nothing** protects you from a question you did not mean to ask

Only the two-question diagnostic from question 7 does that.

**The recap, in one table:**

| | |
|---|---|
| grain is a claim | 5 lines of `is_unique` before any `merge` |
| `LineID` not unique | 40 re-delivered rows, **62,694.27 (0.462%)** of phantom revenue |
| `validate="many_to_one"` | passes on all three dimension joins — checks shape, not meaning |
| the fan-out | 558 returns → **837 rows**, 1.50 per return |
| the count | overstated by **50%** |
| the sum | **identical**, 1,485,707.73 — `Sales` did not fan out |
| the rule | *a fan-out damages a measure only if the measure comes from the side that fanned out* |
| `inner` vs `left` | agree at 8,060 — evidence, not safety |

Sheet 02 takes the other half of the week's theme: not rows that multiply, but
rows that quietly disappear.